# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imnxr/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Data Loading and Modeling Setup

In [21]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

data_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Current-window proxy label supplied by the starter-data design.
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

print("Shape:", df.shape)
print("Clients:", df["client_id"].nunique())
print("Decline base rate:", round(df["is_declining_label"].mean(), 4))

Shape: (30000, 45)
Clients: 32
Decline base rate: 0.5421


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice

I use Logistic Regression as the first learned model.

The target is binary: whether a content item is currently classified as declining. Logistic Regression fits this yes/no question, produces probability scores that can rank content for review, and is easier to interpret than a complex ensemble.

This model is intended as decision support rather than an automatic content decision. I will compare it with the Week 4 rule baseline using the same held-out clients and the same metrics.

The label is a current-window proxy derived from `trend_direction`. To avoid direct leakage, neither `trend_direction` nor `trend_pct` is included as a feature.

In [22]:
# Declare the modeling target and leakage boundaries.

target_column = "is_declining_label"

forbidden_features = [
    "content_id",          # identifier, not a predictive feature
    "client_id",           # used only for grouped splitting
    "trend_direction",     # directly defines the target
    "trend_pct",           # used to compute trend_direction
    target_column,
]

print("Method: Logistic Regression")
print("Target:", target_column)
print("Forbidden model features:")
for column in forbidden_features:
    print("-", column)


Method: Logistic Regression
Target: is_declining_label
Forbidden model features:
- content_id
- client_id
- trend_direction
- trend_pct
- is_declining_label


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

I use a grouped train/test split based on `client_id`.

This is more honest than a random row split because rows from the same client may share patterns, content strategy, measurement setup, and data quality. Keeping each client entirely in either train or test reduces the risk that the model learns client-specific behavior and then appears stronger by seeing the same client during evaluation.

The test set contains approximately 25% of clients. The random seed is fixed for reproducibility.

In [23]:
# Grouped split: each client belongs entirely to train or test.

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df[target_column],
        groups=df["client_id"],
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Train base rate:", round(train_df[target_column].mean(), 4))
print("Test base rate:", round(test_df[target_column].mean(), 4))

Train rows: 22885
Test rows: 7115
Train clients: 24
Test clients: 8
Client overlap: 0
Train base rate: 0.55
Test base rate: 0.5165


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [24]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

y_train = train_df[target_column]
y_test = test_df[target_column]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features before encoding:", len(feature_columns))
print("Train matrix shape:", X_train.shape)
print("Test matrix shape:", X_test.shape)

Numeric features: 23
Categorical features: 9
Total features before encoding: 32
Train matrix shape: (22885, 32)
Test matrix shape: (7115, 32)


In [25]:
# Preprocessing pipelines

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.5).astype(int)

print("Model trained successfully.")

Model trained successfully.


In [26]:
# Honest rule baseline using only safe, non-label-derived signals.

honest_baseline_score = (
    0.40 * (test_df["engagement_rate"].fillna(0) < 1).astype(float)
    + 0.30 * (test_df["avg_position"].replace(0, np.nan).fillna(100) > 20).astype(float)
    + 0.30 * (test_df["days_since_last_update"].fillna(0) >= 90).astype(float)
)

honest_baseline_pred = (honest_baseline_score >= 0.5).astype(int)

honest_results = pd.DataFrame(
    [
        metric_row(
            "Honest rule baseline",
            y_test,
            honest_baseline_pred,
            honest_baseline_score,
        ),
        metric_row(
            "Logistic Regression",
            y_test,
            model_pred,
            model_prob,
        ),
    ]
)

honest_results.insert(
    1,
    "test_base_rate",
    y_test.mean(),
)

honest_results.round(4)

,method,test_base_rate,accuracy,precision,recall,f1,roc_auc
0,Honest rule baseline,0.5165,0.4720,0.4808,0.2792,0.3532,0.4963
1,Logistic Regression,0.5165,0.5619,0.5555,0.7603,0.6419,0.5816


### Model versus honest baseline

Logistic Regression performs better than the honest rule baseline on the same held-out clients and metrics.

The model improves F1 score from 0.3532 to 0.6419 and recall from 0.2792 to 0.7603. This means it detects substantially more of the observed declining pages.

However, the model's ROC-AUC is 0.5816, only moderately above random ranking. The available safe features therefore contain some directional information, but they do not separate declining and non-declining content strongly.

An earlier comparison using the Week 4 operational rule was rejected as a predictive comparison because that rule used `trend_direction`, the field from which the target is created.

## Fair baseline comparison

The original Week 4 operational rule used `trend_direction`. For this Week 5 target, that would be direct leakage because the label is defined from the same field.

Therefore, I do not use the original rule as the headline predictive comparison. I recreate an honest transparent baseline using only engagement rate, average position, and content staleness. Logistic Regression and this honest baseline are evaluated on the same held-out clients and with the same metrics.

The original Week 4 rule remains useful as an operational ranking rule, but its result is not treated as evidence of predictive performance for this label.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and interpretation

The Logistic Regression model identifies many declining pages, but its precision is limited.

It produced 2,236 false positives and 881 false negatives. Its recall of 0.7603 means it catches most declining pages, while its precision of 0.5555 means that a substantial number of review recommendations would be unnecessary.

The ROC-AUC of 0.5816 shows that the model's ranking ability is weak to moderate. Its probabilities should therefore be treated as decision-support signals, not confident predictions or automatic refresh decisions.

In [27]:
# Confusion matrix and error counts

cm = confusion_matrix(y_test, model_pred)

tn, fp, fn, tp = cm.ravel()

error_summary = pd.DataFrame(
    {
        "count": [tn, fp, fn, tp]
    },
    index=[
        "True negatives",
        "False positives",
        "False negatives",
        "True positives",
    ],
)

error_summary

,count
True negatives,1204
False positives,2236
False negatives,881
True positives,2794


In [28]:
error_cases = test_df[
    [
        "content_id",
        "content_type",
        "main_intent",
        "engagement_rate",
        "avg_position",
        "days_since_last_update",
        target_column,
    ]
].copy()

error_cases["predicted_label"] = model_pred
error_cases["decline_probability"] = model_prob

false_positives = error_cases[
    (error_cases[target_column] == 0)
    & (error_cases["predicted_label"] == 1)
].sort_values("decline_probability", ascending=False)

false_negatives = error_cases[
    (error_cases[target_column] == 1)
    & (error_cases["predicted_label"] == 0)
].sort_values("decline_probability", ascending=True)

print("Top false positives:")
display(false_positives.head(3))

print("Top false negatives:")
display(false_negatives.head(3))

Top false positives:


,content_id,content_type,main_intent,engagement_rate,avg_position,days_since_last_update,is_declining_label,predicted_label,decline_probability
10175,content_374e795aab68,keyword article,commercial,0.0,31.0,20,0,1,0.940025
26614,content_7be5f150dc65,keyword article,informational,0.0,5.9,20,0,1,0.925364
20736,content_41baf0722ad9,keyword article,informational,0.0,12.8,104,0,1,0.904988


Top false negatives:


,content_id,content_type,main_intent,engagement_rate,avg_position,days_since_last_update,is_declining_label,predicted_label,decline_probability
12845,content_742a8fcba2fe,keyword article,informational,0.0,76.0,20,1,0,0.077603
4081,content_917fc1b11fe1,keyword article,informational,0.0,78.6,22,1,0,0.077715
29158,content_e18144cbd19d,keyword article,informational,0.0,2.0,20,1,0,0.077788


### Concrete error review

The corrected model produced 2,236 false positives and 881 false negatives.

The false positives include pages with recent updates or comparatively strong search positions. This suggests that the model sometimes interprets broader traffic and content patterns as decline risk even when individual indicators appear healthy.

The false negatives show that declining pages can still have excellent or very poor average positions. Current ranking position alone therefore does not reliably describe recent movement.

These errors support using the predicted probability as a review-priority signal rather than an automatic refresh decision.

In [29]:
# Inspect the strongest Logistic Regression coefficients.

fitted_preprocessor = model.named_steps["preprocessor"]
fitted_classifier = model.named_steps["classifier"]

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_classifier.coef_[0]

coefficient_table = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
    }
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

top_positive = coefficient_table.sort_values(
    "coefficient",
    ascending=False,
).head(10)

top_negative = coefficient_table.sort_values(
    "coefficient",
    ascending=True,
).head(10)

print("Features most associated with predicted decline:")
display(top_positive[["feature", "coefficient"]])

print("Features most associated with predicted non-decline:")
display(top_negative[["feature", "coefficient"]])

Features most associated with predicted decline:


,feature,coefficient
8,num__sessions_90d,0.917793
13,num__days_with_impressions,0.794017
46,cat__word_count_tier_1000-2000,0.547990
42,cat__freshness_tier_0-30,0.384573
61,cat__position_tier_striking,0.337863
3,num__word_count,0.313692
60,cat__position_tier_page_3_5,0.267775
56,cat__impression_tier_low,0.244955
17,num__days_since_last_update,0.239743
58,cat__position_tier_deep,0.221145


Features most associated with predicted non-decline:


,feature,coefficient
9,num__users_90d,-1.034297
62,cat__position_tier_top_3,-1.026926
14,num__days_with_sessions,-0.538061
43,cat__freshness_tier_181+,-0.396214
36,cat__main_intent_navigational,-0.354276
57,cat__impression_tier_moderate,-0.345098
48,cat__word_count_tier_3500+,-0.325406
15,num__content_age_days,-0.317705
19,num__avg_position,-0.295947
47,cat__word_count_tier_2000-3500,-0.249408


### Feature interpretation

Higher `sessions_90d`, more days with impressions, some position tiers, and longer time since update were associated with a higher predicted probability of decline.

Higher `users_90d`, top-three ranking status, more days with sessions, and some longer-form content tiers were associated with a lower predicted probability of decline.

These coefficients describe associations within this fitted model; they do not prove that any feature causes decline. Because several features are correlated and categorical variables are one-hot encoded, coefficient size should be interpreted cautiously.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.